# Q1: Machine Translation with BLEU Score Evaluation

This notebook translates text using Helsinki-NLP's OPUS-MT models and evaluates the translation quality using BLEU score.

**Instructions:**
1. Upload your `input.txt` (source language sentences) and `reference.txt` (reference translations) files
2. Set the correct model name for your language pair
3. Run all cells to get translations and BLEU score

## Step 1: Install Dependencies

In [ ]:
!pip install torch transformers huggingface_hub sacrebleu accelerate sentencepiece sacremoses -q

## Step 2: Upload Input and Reference Files

Run this cell and upload:
- `input.txt` - Source language sentences (one per line)
- `reference.txt` - Reference translations (one per line)

In [ ]:
from google.colab import files

print("Please upload input.txt and reference.txt files:")
uploaded = files.upload()

## Step 3: Configuration

Set your model name based on the source and target language pair.

Examples:
- Bengali to English: `Helsinki-NLP/opus-mt-bn-en` (default for this assignment)
- Hindi to English: `Helsinki-NLP/opus-mt-hi-en`
- French to English: `Helsinki-NLP/opus-mt-fr-en`
- German to English: `Helsinki-NLP/opus-mt-de-en`

In [ ]:
# Configuration
MODEL_NAME = "Helsinki-NLP/opus-mt-bn-en"  # Bengali to English
INPUT_FILE = "input.txt"
REFERENCE_FILE = "reference.txt"
OUTPUT_FILE = "output.txt"
BATCH_SIZE = 16

## Step 4: Define the Translator Class

In [ ]:
import torch
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class Translator:
    def __init__(self, model_name: str, batch_size: int = 16):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        
        print(f"Loading model: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        self.batch_size = batch_size
        print("Model loaded successfully!")

    def _read_file(self, filepath: str) -> list:
        with open(filepath, "r", encoding="utf-8") as file:
            return [line.strip() for line in file if line.strip()]

    def _write_file(self, lines: list, filepath: str) -> None:
        with open(filepath, "w", encoding="utf-8") as file:
            file.write("\n".join(lines) + "\n")

    def _translate_batch(self, texts: list) -> list:
        outputs = []
        total_batches = (len(texts) + self.batch_size - 1) // self.batch_size
        
        for i in range(0, len(texts), self.batch_size):
            batch_num = i // self.batch_size + 1
            print(f"Translating batch {batch_num}/{total_batches}...")
            
            batch = texts[i:i + self.batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
            
            with torch.no_grad():
                generated_tokens = self.model.generate(**inputs)
            
            decoded = self.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            outputs.extend(decoded)
            
        return outputs

    def evaluate_model(self, input_path: str, ref_path: str, output_path: str) -> float:
        print(f"\nReading source texts from: {input_path}")
        source_texts = self._read_file(input_path)
        print(f"Found {len(source_texts)} sentences to translate")
        
        print(f"Reading reference translations from: {ref_path}")
        reference_texts = self._read_file(ref_path)
        print(f"Found {len(reference_texts)} reference translations")
        
        print("\nStarting translation...")
        translated_texts = self._translate_batch(source_texts)
        
        print(f"\nSaving translations to: {output_path}")
        self._write_file(translated_texts, output_path)
        
        print("\nCalculating BLEU score...")
        bleu = sacrebleu.corpus_bleu(translated_texts, [reference_texts])
        return bleu.score, translated_texts, reference_texts

## Step 5: Run Translation and Evaluation

In [ ]:
# Initialize translator
translator = Translator(MODEL_NAME, BATCH_SIZE)

# Run translation and evaluation
bleu_score, translations, references = translator.evaluate_model(INPUT_FILE, REFERENCE_FILE, OUTPUT_FILE)

print("\n" + "="*50)
print(f"BLEU:{bleu_score:.2f}")
print("="*50)

## Step 6: View Results

In [ ]:
# Display sample translations
print("\n" + "="*80)
print("SAMPLE TRANSLATIONS (first 5)")
print("="*80)

for i, (trans, ref) in enumerate(zip(translations[:5], references[:5]), 1):
    print(f"\n[{i}] Translation: {trans}")
    print(f"    Reference:   {ref}")

In [ ]:
# Download the output file
from google.colab import files
files.download(OUTPUT_FILE)

## (Optional) View All Translations

In [ ]:
# View all translations
print("\nALL TRANSLATIONS:")
print("="*80)
for i, (trans, ref) in enumerate(zip(translations, references), 1):
    print(f"\n[{i}] Translation: {trans}")
    print(f"    Reference:   {ref}")